# Multi-Agent Condition Pipeline

**What you'll build:** A two-agent workflow where Agent 1 extracts switch variables and Agent 2 generates a conditional answer — with conditions passed between agents as structured JSON.

**Time:** 15 minutes

**Learning objectives:**
1. Implement the Condition Extractor → Conditional Answerer pipeline
2. Pass switch variables between agents without loss using structured JSON
3. See how condition passing prevents the decay problem from Guide 01
4. Build a condition handoff that survives a multi-step chain

**Prerequisites:**
- Notebook 01 (condition-aware single agent)
- Guide 01 (condition decay and structured handoffs)
- Guide 02 (Claude-specific patterns)

In [ ]:
learning_objectives(["A two-agent workflow where Agent 1 extracts switch variables and Agent 2 generates a conditional answer — with conditions passed between agents as structured JSON", "15 minutes", "1. Implement the Condition Extractor → Conditional Answerer pipeline", "Pass switch variables between agents without loss using structured JSON", "See how condition passing prevents the decay problem from Guide 01", "Build a condition handoff that survives a multi-step chain"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

## The Architecture

```
User Question
     │
     ▼
┌────────────────────────────────────┐
│  Agent 1: Condition Extractor      │
│  - Analyzes the question           │
│  - Identifies switch variables     │
│  - Classifies each as present/     │
│    missing/inferred                │
│  Output: structured JSON           │
└────────────────┬───────────────────┘
                 │  JSON handoff
                 │  (switch variables + condition stack)
                 ▼
┌────────────────────────────────────┐
│  Agent 2: Conditional Answerer     │
│  - Receives conditions as JSON     │
│  - Generates answer for each       │
│    branch of the switch variable   │
│  - Produces a decision tree answer │
│  Output: conditional answer JSON   │
└────────────────────────────────────┘
```

The critical difference from a naive pipeline: the handoff is **structured data**, not prose.

In [ ]:
import anthropic
import json
from dataclasses import dataclass, asdict
from typing import Optional

client = anthropic.Anthropic()  # Uses ANTHROPIC_API_KEY from environment

# The question we'll use throughout this notebook
# A question that genuinely has different answers depending on conditions
QUESTION = "Should we build this feature in-house or buy a third-party solution?"

print("Question:", QUESTION)
print("\nThis question has at least 4 switch variables that flip the recommendation.")
print("Let's use two Claude agents to identify them and generate conditional answers.")

## Define the Handoff Schema

The handoff schema is the contract between Agent 1 and Agent 2. Both agents must agree on it before we write any prompts.

In [ ]:
# The JSON schema that Agent 1 produces and Agent 2 consumes
# This is the condition payload — the data that must survive the handoff

HANDOFF_SCHEMA = {
    "type": "object",
    "required": [
        "question_summary",
        "switch_variables",
        "condition_stack",
        "recommended_answer_structure"
    ],
    "properties": {
        "question_summary": {
            "type": "string",
            "description": "One sentence summary of what the question is asking"
        },
        "switch_variables": {
            "type": "array",
            "description": "Conditions that flip the correct answer",
            "items": {
                "type": "object",
                "required": ["name", "description", "possible_values", "status"],
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Short identifier (snake_case)"
                    },
                    "description": {
                        "type": "string",
                        "description": "What this variable measures and why it matters"
                    },
                    "possible_values": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "The main branches this variable creates"
                    },
                    "status": {
                        "type": "string",
                        "enum": ["present", "missing", "inferred"],
                        "description": "Whether this condition is specified, absent, or can be inferred from context"
                    },
                    "current_value": {
                        "type": "string",
                        "description": "The value if status is 'present' or 'inferred'"
                    }
                }
            }
        },
        "condition_stack": {
            "type": "object",
            "description": "The 6-layer condition stack as extracted from the question",
            "properties": {
                "layer_1_domain":      {"type": "string"},
                "layer_2_context":     {"type": "string"},
                "layer_3_objective":   {"type": "string"},
                "layer_4_constraints": {"type": "array", "items": {"type": "string"}},
                "layer_5_facts":       {"type": "string"},
                "layer_6_output_spec": {"type": "string"}
            }
        },
        "recommended_answer_structure": {
            "type": "string",
            "enum": ["single_answer", "conditional_tree", "comparison_table"],
            "description": "What structure Agent 2 should use for its answer"
        },
        "primary_switch_variable": {
            "type": "string",
            "description": "The single most important switch variable — the one that most changes the answer"
        }
    }
}

print("Handoff schema defined.")
print(f"Agent 1 will produce: {list(HANDOFF_SCHEMA['properties'].keys())}")

## Agent 1: Condition Extractor

Agent 1's job: analyze the question and produce the handoff payload.

In [ ]:
EXTRACTOR_SYSTEM = """You are a decision analysis specialist.

Role: Analyze questions and identify the switch variables — the conditions that, 
if changed, would lead to a different recommendation.

Objective: Produce a structured condition payload that a separate answering agent 
can use to generate a precise conditional answer.

Standards:
- Identify 2-5 switch variables (not an exhaustive list)
- Rank them by how much they affect the answer
- For each, specify the possible values that create different answer branches
- Extract whatever condition stack layers are present in the question
- Recommend the answer structure that fits the number of missing switch variables"""


def run_extractor_agent(question: str) -> dict:
    """
    Agent 1: Extract switch variables and condition stack from a question.
    
    Returns the handoff payload that Agent 2 will consume.
    """
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=2000,
        system=EXTRACTOR_SYSTEM,
        tools=[{
            "name": "submit_handoff_payload",
            "description": "Submit the structured condition payload for the answering agent.",
            "input_schema": HANDOFF_SCHEMA
        }],
        # Force structured output — Agent 2 needs machine-parseable JSON, not prose
        tool_choice={"type": "tool", "name": "submit_handoff_payload"},
        messages=[{
            "role": "user",
            "content": f"""Analyze this question and produce a condition payload for the answering agent:

{question}"""
        }]
    )
    
    for block in response.content:
        if block.type == "tool_use" and block.name == "submit_handoff_payload":
            return block.input
    
    raise ValueError("Extractor agent did not return structured handoff payload")


# Run Agent 1
print("Running Agent 1: Condition Extractor...")
handoff_payload = run_extractor_agent(QUESTION)

print("\n=== Agent 1 Output (Handoff Payload) ===")
print(f"Question summary: {handoff_payload['question_summary']}")
print(f"\nSwitch variables ({len(handoff_payload['switch_variables'])} found):")
for sv in handoff_payload["switch_variables"]:
    status_marker = "[PRESENT]" if sv["status"] == "present" else "[MISSING]"
    print(f"  {status_marker} {sv['name']}: {sv['description']}")
    print(f"    Possible values: {sv['possible_values']}")

print(f"\nPrimary switch variable: {handoff_payload.get('primary_switch_variable', 'not specified')}")
print(f"Recommended answer structure: {handoff_payload['recommended_answer_structure']}")

print("\nCondition stack:")
for layer, value in handoff_payload["condition_stack"].items():
    if value:
        print(f"  {layer}: {value}")

## Agent 2: Conditional Answerer

Agent 2 receives the handoff payload and generates an answer conditioned on it.

If switch variables are missing, it generates a **conditional tree** — one answer branch per possible value.

In [ ]:
# Output schema for Agent 2
# The final structured answer — conditioned on the received switch variables

ANSWER_SCHEMA = {
    "type": "object",
    "required": ["answer_structure", "answer", "switch_variables_consumed"],
    "properties": {
        "answer_structure": {
            "type": "string",
            "enum": ["single_answer", "conditional_tree", "comparison_table"]
        },
        "answer": {
            "type": "object",
            "description": "The answer content — structure depends on answer_structure",
            "properties": {
                # For single_answer
                "recommendation": {
                    "type": "string",
                    "description": "Direct recommendation (used when all switch variables are present)"
                },
                "reasoning": {
                    "type": "string",
                    "description": "Explanation referencing specific conditions"
                },
                # For conditional_tree
                "branches": {
                    "type": "array",
                    "description": "One branch per combination of missing switch variable values",
                    "items": {
                        "type": "object",
                        "properties": {
                            "condition_values": {
                                "type": "object",
                                "description": "The switch variable values that define this branch"
                            },
                            "recommendation": {"type": "string"},
                            "key_reason":     {"type": "string"}
                        }
                    }
                }
            }
        },
        "switch_variables_consumed": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Names of switch variables that changed the answer"
        },
        "confidence": {
            "type": "string",
            "enum": ["high", "medium", "low"]
        },
        "conditions_that_would_sharpen_answer": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Missing conditions that, if specified, would increase confidence"
        }
    }
}


ANSWERER_SYSTEM = """You are a conditional reasoning specialist.

Role: Generate answers that are precisely conditioned on specified switch variables.

Objective: When switch variables are present, give a specific recommendation.
When switch variables are missing, generate a conditional tree — one branch per 
combination of the most important missing variable's values.

Standard: Never answer as if advising the average case. Every recommendation must 
reference the specific conditions that justify it. If a recommendation would change 
if any condition changed, say which condition and how."""


def run_answerer_agent(question: str, handoff_payload: dict) -> dict:
    """
    Agent 2: Generate a conditional answer using the handoff payload from Agent 1.
    
    Injects the condition payload into the user message as structured JSON.
    This is the mechanism that prevents condition decay.
    """
    # Format the handoff payload for injection
    # JSON, not prose — conditions cannot be summarized away
    payload_text = json.dumps(handoff_payload, indent=2)
    
    # The user message includes the question AND the condition payload
    # The payload is explicit structured data, not embedded in natural language
    user_message = f"""Original question: {question}

Condition analysis from extraction agent:
{payload_text}

Generate an answer conditioned on this payload.
- If the primary switch variable has a known value: give a specific recommendation
- If the primary switch variable is missing: generate a conditional tree
  with one branch per possible value
- Reference specific conditions in your reasoning — not generic advice"""
    
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=2500,
        system=ANSWERER_SYSTEM,  # Layer 0: persistent role
        tools=[{
            "name": "submit_conditioned_answer",
            "description": "Submit the structured, conditioned answer.",
            "input_schema": ANSWER_SCHEMA
        }],
        tool_choice={"type": "tool", "name": "submit_conditioned_answer"},
        messages=[{"role": "user", "content": user_message}]
    )
    
    for block in response.content:
        if block.type == "tool_use" and block.name == "submit_conditioned_answer":
            return block.input
    
    raise ValueError("Answerer agent did not return structured output")


# Run Agent 2 with the handoff payload from Agent 1
print("Running Agent 2: Conditional Answerer...")
answer_result = run_answerer_agent(QUESTION, handoff_payload)

print("\n=== Agent 2 Output ===")
print(f"Answer structure: {answer_result['answer_structure']}")
print(f"Confidence: {answer_result['confidence']}")
print(f"Switch variables consumed: {answer_result['switch_variables_consumed']}")

answer = answer_result["answer"]

if answer_result["answer_structure"] == "single_answer":
    print(f"\nRecommendation: {answer.get('recommendation')}")
    print(f"Reasoning: {answer.get('reasoning')}")

elif answer_result["answer_structure"] == "conditional_tree":
    print("\nConditional answer branches:")
    for i, branch in enumerate(answer.get("branches", []), 1):
        print(f"\n  Branch {i}: {branch['condition_values']}")
        print(f"  Recommendation: {branch['recommendation']}")
        print(f"  Key reason: {branch['key_reason']}")

if answer_result.get("conditions_that_would_sharpen_answer"):
    print(f"\nConditions that would increase confidence: {answer_result['conditions_that_would_sharpen_answer']}")

## The Full Pipeline as One Function

Here's the complete two-agent pipeline with condition monitoring:

In [ ]:
def run_multi_agent_pipeline(
    question: str,
    verbose: bool = True
) -> dict:
    """
    Complete two-agent condition passing pipeline.
    
    Agent 1: Extract switch variables → produce handoff payload
    Handoff: Structured JSON (conditions survive intact)
    Agent 2: Receive conditions → generate conditional answer
    
    Returns:
        dict with keys: handoff_payload, answer, pipeline_summary
    """
    if verbose:
        print(f"Question: {question}")
        print("-" * 60)
    
    # Agent 1: Extract conditions
    if verbose:
        print("[Agent 1] Extracting switch variables...")
    handoff = run_extractor_agent(question)
    
    if verbose:
        present = [sv["name"] for sv in handoff["switch_variables"] if sv["status"] == "present"]
        missing = [sv["name"] for sv in handoff["switch_variables"] if sv["status"] == "missing"]
        print(f"  Found {len(handoff['switch_variables'])} switch variables")
        print(f"  Present: {present}")
        print(f"  Missing: {missing}")
        print(f"  Recommended structure: {handoff['recommended_answer_structure']}")
    
    # Handoff: JSON payload passes between agents
    if verbose:
        print("\n[Handoff] Passing condition payload to Agent 2...")
        condition_count = sum(1 for v in handoff["condition_stack"].values() if v)
        print(f"  Condition stack layers filled: {condition_count}/6")
        print(f"  Switch variables in payload: {len(handoff['switch_variables'])}")
    
    # Agent 2: Generate conditioned answer
    if verbose:
        print("\n[Agent 2] Generating conditioned answer...")
    answer = run_answerer_agent(question, handoff)
    
    if verbose:
        print(f"  Answer structure: {answer['answer_structure']}")
        print(f"  Confidence: {answer['confidence']}")
        print(f"  Conditions consumed: {answer['switch_variables_consumed']}")
    
    return {
        "handoff_payload": handoff,
        "answer":          answer,
        "pipeline_summary": {
            "switch_variables_found":    len(handoff["switch_variables"]),
            "switch_variables_present":  len([sv for sv in handoff["switch_variables"] if sv["status"] == "present"]),
            "switch_variables_missing":  len([sv for sv in handoff["switch_variables"] if sv["status"] == "missing"]),
            "answer_structure":          answer["answer_structure"],
            "confidence":                answer["confidence"]
        }
    }


# Test on a second question
result2 = run_multi_agent_pipeline(
    question="When should we migrate from monolith to microservices?"
)

print("\n=== Pipeline Summary ===")
for k, v in result2["pipeline_summary"].items():
    print(f"  {k}: {v}")

## Condition Decay Demo: What Happens Without Structured Handoff

This cell demonstrates what condition decay looks like — and why structured handoffs matter.

In [ ]:
# DEMONSTRATION: Condition decay via natural language handoff
# 
# Compare: what Agent 2 produces when it receives...
# (A) The structured JSON handoff payload
# (B) A natural language summary of that same payload

test_question = "Should we raise prices for our product?"

# Get the structured handoff
structured_handoff = run_extractor_agent(test_question)

# Create a natural language "summary" of the same payload
# This simulates what happens when agents pass prose instead of JSON
nl_summary = f"""Based on the question about pricing, the main considerations are:
- The current pricing strategy and competitive position
- How customers might respond to price changes
- The company's revenue goals
- Market conditions

The question seems to be about whether to raise prices, which depends on various factors."""

# Get answers under both conditions
print("=== Comparing Structured vs. Natural Language Handoffs ===")
print()

# With structured handoff
print("[Structured JSON handoff]")
structured_answer = run_answerer_agent(test_question, structured_handoff)
print(f"  Answer structure: {structured_answer['answer_structure']}")
print(f"  Confidence: {structured_answer['confidence']}")
print(f"  Switch variables consumed: {structured_answer['switch_variables_consumed']}")

print()

# With natural language "handoff" (simulating decay)
print("[Natural language handoff — simulates condition decay]")
nl_response = client.messages.create(
    model="claude-opus-4-5",
    max_tokens=400,
    system=ANSWERER_SYSTEM,
    messages=[{
        "role": "user",
        "content": f"""Original question: {test_question}

Context from previous analysis:
{nl_summary}

Generate an answer."""
    }]
)
nl_answer_text = nl_response.content[0].text
print(f"  Answer (first 300 chars): {nl_answer_text[:300]}...")

print()
print("OBSERVATION:")
print("  Structured handoff: specific switch variables, explicit branching, measured confidence")
print("  Natural language handoff: general advice, vague conditions, no explicit branching")
print("  The natural language version lost the specific switch variable structure.")
print("  This is condition decay in action.")

## Run on Your Own Questions

In [ ]:
# EXPERIMENT 1: A question where you already know what the switch variables should be
# Run the pipeline and verify the agent finds the same ones

your_question = "Should we hire a generalist or a specialist for this role?"

# YOUR CODE: What switch variables do you expect for this question?
# expected_switch_variables = ["role_scope", "team_expertise", "budget", "..."]

result = run_multi_agent_pipeline(your_question)

print("\nSwitch variables found by Agent 1:")
for sv in result["handoff_payload"]["switch_variables"]:
    print(f"  - {sv['name']}: {sv['description']}")

In [ ]:
# EXPERIMENT 2: Pre-specify some switch variables and see how confidence changes
# Manually inject conditions into the handoff payload before sending to Agent 2

# Get the base handoff
base_handoff = run_extractor_agent(your_question)

# Manually set values for the first two missing switch variables
# This simulates having more context available
for sv in base_handoff["switch_variables"][:2]:
    if sv["status"] == "missing" and sv["possible_values"]:
        sv["status"] = "present"
        sv["current_value"] = sv["possible_values"][0]  # Use first possible value
        print(f"Set {sv['name']} = {sv['possible_values'][0]}")

# Run Agent 2 with the enriched handoff
enriched_answer = run_answerer_agent(your_question, base_handoff)
print(f"\nConfidence with enriched conditions: {enriched_answer['confidence']}")
print(f"Answer structure: {enriched_answer['answer_structure']}")

# Compare to base pipeline confidence
base_answer = run_answerer_agent(your_question, run_extractor_agent(your_question))
print(f"Confidence without enrichment: {base_answer['confidence']}")

## Summary

**What you built:** A two-agent pipeline where conditions travel as structured JSON — not prose.

**The architecture:**
- Agent 1 (Condition Extractor): Analyzes the question, identifies switch variables, classifies each as present/missing/inferred, produces a structured handoff payload
- Handoff: JSON object containing switch variables + condition stack — passes between agents without loss
- Agent 2 (Conditional Answerer): Receives the payload, generates answer branched on missing switch variables

**Why structured handoffs matter:**
- Natural language summaries lose switch variable structure (condition decay)
- JSON payloads preserve the exact conditions — Agent 2 receives what Agent 1 found
- Forced `tool_choice` at every step guarantees machine-parseable output

**The connection to Guide 01:**
The condition decay problem from the guide is visible in the comparison cell. Natural language handoffs produce generic advice. Structured handoffs produce conditional trees with explicit branching.

**Next:** `exercises/01_design_agent_prompts.md` — design condition-aware system prompts for real agent roles.